# 02 — Root Cause Analysis
**Purpose:** Reproduce every business finding reported in the README. Each section states the business question, method, code, intermediate output, final output, business interpretation, and an explicit reproduction check.


In [30]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 25)

df = pd.read_excel('/content/superstore_data1.xlsx', sheet_name='Sample - Superstore')
df['Year'] = df['Order Date'].dt.year
df['Quarter'] = df['Order Date'].dt.quarter
print(df.shape)


(9994, 23)


## Section 1 — Q4-2016 vs Q4-2017 Comparison

**Business question:** Did Sales, Profit, and Margin actually move in different directions, and over what period?

**Method:** Aggregate Revenue, Profit, and Margin by Year and Quarter; compare Q4-2016 to Q4-2017 against the full 2014–2017 trend.


In [3]:
def agg(g):
    revenue = g['Sales'].sum()
    profit = g['Profit'].sum()
    return pd.Series({'Revenue': revenue, 'Profit': profit, 'Margin%': profit/revenue*100})

yearly = df.groupby('Year').apply(agg, include_groups=False)
yearly


,Revenue,Profit,Margin%
Year,,,
2014,484247.4981,49543.9741,10.231126
2015,470532.5090,61618.6037,13.095504
2016,609205.5980,81795.1743,13.426530
2017,733215.2552,93439.2696,12.743771


In [4]:
q4_16 = df[(df['Year']==2016)&(df['Quarter']==4)]
q4_17 = df[(df['Year']==2017)&(df['Quarter']==4)]

R16, P16 = q4_16['Sales'].sum(), q4_16['Profit'].sum()
R17, P17 = q4_17['Sales'].sum(), q4_17['Profit'].sum()
M16, M17 = P16/R16*100, P17/R17*100

comparison = pd.DataFrame({
    'Metric': ['Revenue','Profit','Margin%'],
    'Q4-2016': [R16, P16, M16],
    'Q4-2017': [R17, P17, M17]
})
comparison


,Metric,Q4-2016,Q4-2017
0,Revenue,236098.753800,280054.067000
1,Profit,38139.859300,27448.726000
2,Margin%,16.154198,9.801224


**Full-trend check (2014 vs 2017):** confirms the pattern does NOT hold across the full period — used to support the README statement "the full 2014–2017 trend shows margin improving."


In [5]:
full_trend = yearly.loc[[2014,2017]]
full_trend['Margin_change_pp'] = full_trend['Margin%'].diff()
full_trend


,Revenue,Profit,Margin%,Margin_change_pp
Year,,,,
2014,484247.4981,49543.9741,10.231126,NaN
2017,733215.2552,93439.2696,12.743771,2.512644


**⚠ Discrepancy identified during reproduction — reported, not corrected:**

The full-year comparison (2016 vs 2017) satisfies the original premise: Sales↑, Profit↑, Margin↓.

However, the Q4-2016 vs Q4-2017 slice — which is the baseline used for every subsequent section in this
notebook (Sections 2–10), consistent with the frozen project's Phase 3–9 methodology — shows Sales↑ but
**Profit↓** ($38,139.86 → $27,448.73) alongside Margin↓. Profit does not increase within this specific
quarterly slice, even though it does increase on a full-year basis.

This means the phrase "Sales↑ Profit↑ Margin↓" in the README's Executive Summary is only fully accurate
for the full-year comparison, not for the Q4-specific comparison that the rest of the root cause analysis
is built on. This is flagged here as a **traceability discrepancy**, not silently resolved, per the
requirement to state — rather than replace — any claim that cannot be reproduced as written.


In [6]:
readme_check_1a = (R17 > R16) and (M17 < M16) and (yearly.loc[2017,'Margin%'] > yearly.loc[2014,'Margin%'])
readme_check_1b_profit_up_in_q4 = P17 > P16

print("Sales up in Q4 (R17 > R16):", R17 > R16)
print("Margin down in Q4 (M17 < M16):", M17 < M16)
print("Profit UP in Q4 (P17 > P16):", readme_check_1b_profit_up_in_q4, "<- does NOT hold")
print("Full-year pattern (2017 margin > 2014 margin):", yearly.loc[2017,'Margin%'] > yearly.loc[2014,'Margin%'])
print()
print("README claim reproduced (full-year framing 'Sales up, Profit up, Margin down'):", "PARTIAL -- holds full-year, NOT within the Q4-only slice used downstream")


Sales up in Q4 (R17 > R16): True
Margin down in Q4 (M17 < M16): True
Profit UP in Q4 (P17 > P16): False <- does NOT hold
Full-year pattern (2017 margin > 2014 margin): True

README claim reproduced (full-year framing 'Sales up, Profit up, Margin down'): PARTIAL -- holds full-year, NOT within the Q4-only slice used downstream


## Section 2 — Revenue Effect vs Margin-Rate Effect

**Business question:** Is the profit decline caused by weaker sales, or by a drop in margin rate?

**Method:** Two-way profit bridge decomposition. Revenue Effect = (R17-R16) × M16(decimal). Margin-Rate Effect = R17 × (M17(decimal)-M16(decimal)).


In [7]:
M16_dec, M17_dec = P16/R16, P17/R17
total_delta = P17 - P16
revenue_effect = (R17 - R16) * M16_dec
margin_effect = R17 * (M17_dec - M16_dec)

bridge = pd.DataFrame({
    'Component': ['Total Profit Delta','Revenue Effect','Margin-Rate Effect'],
    'Dollar': [total_delta, revenue_effect, margin_effect]
})
bridge['% of Total Decline'] = bridge['Dollar']/total_delta*100
bridge


,Component,Dollar,% of Total Decline
0,Total Profit Delta,-10691.133300,100.000000
1,Revenue Effect,7100.628165,-66.416047
2,Margin-Rate Effect,-17791.761465,166.416047


In [8]:
print("Check: Revenue Effect + Margin Effect =", round(revenue_effect+margin_effect,2), "vs Total Delta =", round(total_delta,2))
readme_check_2 = abs((revenue_effect+margin_effect) - total_delta) < 1
print("README claim reproduced:", "YES" if readme_check_2 else "NO")


Check: Revenue Effect + Margin Effect = -10691.13 vs Total Delta = -10691.13
README claim reproduced: YES


## Section 3 — Category Mix Analysis

**Business question:** Is the margin decline explained by a shift toward lower-margin product categories?

**Method:** Shift-share decomposition — Mix Effect (share shift at constant margin) vs Rate Effect (margin shift at constant share).


In [9]:
cat16 = q4_16.groupby('Category').agg(Rev=('Sales','sum'), Prof=('Profit','sum'))
cat17 = q4_17.groupby('Category').agg(Rev=('Sales','sum'), Prof=('Profit','sum'))
cat16['Share'] = cat16['Rev']/cat16['Rev'].sum()
cat16['Margin'] = cat16['Prof']/cat16['Rev']
cat17['Share'] = cat17['Rev']/cat17['Rev'].sum()
cat17['Margin'] = cat17['Prof']/cat17['Rev']

category_mix_table = pd.DataFrame({
    'Share2016%': cat16['Share']*100, 'Share2017%': cat17['Share']*100,
    'Margin2016%': cat16['Margin']*100, 'Margin2017%': cat17['Margin']*100
})
category_mix_table


,Share2016%,Share2017%,Margin2016%,Margin2017%
Category,,,,
Furniture,34.025983,32.261003,4.361516,-1.078163
Office Supplies,31.667528,30.332168,27.864900,11.094270
Technology,34.306490,37.406829,17.040578,18.135505


In [10]:
actual16 = (cat16['Share']*cat16['Margin']).sum()
actual17 = (cat17['Share']*cat17['Margin']).sum()
mix_effect = (cat17['Share']*cat16['Margin']).sum() - actual16
rate_effect = (cat16['Share']*cat17['Margin']).sum() - actual16
interaction = (actual17-actual16) - mix_effect - rate_effect

category_bridge = pd.DataFrame({
    'Component': ['Mix Effect','Rate Effect','Interaction'],
    'pp': [mix_effect*100, rate_effect*100, interaction*100],
    'Dollar (x R17)': [mix_effect*R17, rate_effect*R17, interaction*R17]
})
category_bridge


,Component,pp,Dollar (x R17)
0,Mix Effect,0.079239,221.912955
1,Rate Effect,-6.786117,-19004.796530
2,Interaction,0.353904,991.122110


In [11]:
print("Category Mix Effect is positive (%.2f pp) -> mix shift moved in a margin-favorable direction, not unfavorable." % (mix_effect*100))
readme_check_3 = mix_effect > 0
print("README claim reproduced:", "YES" if readme_check_3 else "NO")


Category Mix Effect is positive (0.08 pp) -> mix shift moved in a margin-favorable direction, not unfavorable.
README claim reproduced: YES


## Section 4 — Region Mix Analysis

**Business question:** Did a shift in regional revenue share contribute to the margin decline?

**Method:** Same shift-share decomposition applied at Region level.


In [12]:
reg16 = q4_16.groupby('Region').agg(Rev=('Sales','sum'), Prof=('Profit','sum'))
reg17 = q4_17.groupby('Region').agg(Rev=('Sales','sum'), Prof=('Profit','sum'))
reg16['Share'] = reg16['Rev']/reg16['Rev'].sum()
reg16['Margin'] = reg16['Prof']/reg16['Rev']
reg17['Share'] = reg17['Rev']/reg17['Rev'].sum()
reg17['Margin'] = reg17['Prof']/reg17['Rev']

region_mix_table = pd.DataFrame({
    'Share2016%': reg16['Share']*100, 'Share2017%': reg17['Share']*100,
    'Margin2016%': reg16['Margin']*100, 'Margin2017%': reg17['Margin']*100
})
region_mix_table


,Share2016%,Share2017%,Margin2016%,Margin2017%
Region,,,,
Central,28.834540,16.482669,28.662472,-4.678146
East,27.824270,35.001547,6.648048,20.661666
South,12.562036,20.019030,17.102137,-2.748183
West,30.779154,28.496754,12.642837,13.652657


In [13]:
ractual16 = (reg16['Share']*reg16['Margin']).sum()
ractual17 = (reg17['Share']*reg17['Margin']).sum()
rmix_effect = (reg17['Share']*reg16['Margin']).sum() - ractual16
rrate_effect = (reg16['Share']*reg17['Margin']).sum() - ractual16
rinteraction = (ractual17-ractual16) - rmix_effect - rrate_effect

region_bridge = pd.DataFrame({
    'Component': ['Mix Effect','Rate Effect','Interaction'],
    'pp': [rmix_effect*100, rrate_effect*100, rinteraction*100],
    'Dollar (x R17)': [rmix_effect*R17, rrate_effect*R17, rinteraction*R17]
})
region_bridge


,Component,pp,Dollar (x R17)
0,Mix Effect,-2.076457,-5815.203502
1,Rate Effect,-7.897217,-22116.477972
2,Interaction,3.620701,10139.920009


In [14]:
profit_delta_by_region = (reg17['Prof'] - reg16['Prof']).sort_values()
profit_delta_by_region


,Prof
Region,
Central,-21672.2846
South,-6613.0351
West,1708.2344
East,15885.9520


In [15]:
print("Region Mix Effect: %.2f pp (%.2f USD) -- negative, unlike Category Mix" % (rmix_effect*100, rmix_effect*R17))
readme_check_4 = rmix_effect < 0
print("README claim reproduced:", "YES" if readme_check_4 else "NO")


Region Mix Effect: -2.08 pp (-5815.20 USD) -- negative, unlike Category Mix
README claim reproduced: YES


## Section 5 — Binders Analysis (Sub-Category Drill-Down)

**Business question:** Which sub-category contributes most to the profit decline?

**Method:** Profit-delta decomposition by Sub-Category, Q4-2016 vs Q4-2017.


In [16]:
sub16 = q4_16.groupby('Sub-Category').agg(Prof16=('Profit','sum'))
sub17 = q4_17.groupby('Sub-Category').agg(Prof17=('Profit','sum'))
subcat_delta = sub16.join(sub17, how='outer').fillna(0)
subcat_delta['ProfitDelta'] = subcat_delta['Prof17'] - subcat_delta['Prof16']
subcat_delta = subcat_delta.sort_values('ProfitDelta')
subcat_delta


,Prof16,Prof17,ProfitDelta
Sub-Category,,,
Binders,10488.8960,-2198.7095,-12687.6055
Tables,-1551.5713,-4530.3769,-2978.8056
Bookcases,17.9748,-893.7648,-911.7396
Appliances,3063.5178,2466.7066,-596.8112
Furnishings,2155.9012,1578.1652,-577.7360
Copiers,12161.9082,11887.9254,-273.9828
Envelopes,739.1725,596.7199,-142.4526
Labels,510.1852,402.5490,-107.6362
Chairs,2881.5156,2871.8747,-9.6409


In [17]:
binders_delta = subcat_delta.loc['Binders','ProfitDelta']
print("Binders ProfitDelta: $%.2f (%.1f%% of total decline)" % (binders_delta, binders_delta/total_delta*100))
print("Sum of all sub-category deltas:", round(subcat_delta['ProfitDelta'].sum(),2), " vs Total Delta:", round(total_delta,2))
readme_check_5 = abs(binders_delta - (-12687.61)) < 1
print("README claim reproduced:", "YES" if readme_check_5 else "NO")


Binders ProfitDelta: $-12687.61 (118.7% of total decline)
Sum of all sub-category deltas: -10691.13  vs Total Delta: -10691.13
README claim reproduced: YES


## Section 6 — High-Ticket Transaction Analysis (Binders Cluster)

**Business question:** Is the Binders decline caused by the whole sub-category, or a small transaction cluster?

**Method:** Split Binders transactions into high-ticket (Sales > $500 per line) and the remainder; compare profit delta and discount depth.


In [18]:
b16 = q4_16[q4_16['Sub-Category']=='Binders']
b17 = q4_17[q4_17['Sub-Category']=='Binders']

high16 = b16[b16['Sales']>500]
high17 = b17[b17['Sales']>500]
rest16 = b16[b16['Sales']<=500]
rest17 = b17[b17['Sales']<=500]

binders_cluster = pd.DataFrame({
    'Segment': ['High-ticket (Sales>500)','Rest of Binders'],
    'N_2016': [len(high16), len(rest16)],
    'N_2017': [len(high17), len(rest17)],
    'Profit_2016': [high16['Profit'].sum(), rest16['Profit'].sum()],
    'Profit_2017': [high17['Profit'].sum(), rest17['Profit'].sum()],
    'AvgDiscount_2016%': [high16['Discount'].mean()*100, rest16['Discount'].mean()*100],
    'AvgDiscount_2017%': [high17['Discount'].mean()*100, rest17['Discount'].mean()*100],
})
binders_cluster['ProfitDelta'] = binders_cluster['Profit_2017'] - binders_cluster['Profit_2016']
binders_cluster


,Segment,N_2016,N_2017,Profit_2016,Profit_2017,AvgDiscount_2016%,AvgDiscount_2017%,ProfitDelta
0,High-ticket (Sales>500),8,10,9063.3862,-3194.9170,12.500000,38.000000,-12258.3032
1,Rest of Binders,147,175,1425.5098,996.2075,33.537415,36.857143,-429.3023


In [19]:
high_ticket_share_of_binders = binders_cluster.loc[0,'ProfitDelta'] / binders_delta * 100
print("High-ticket cluster share of total Binders decline: %.1f%%" % high_ticket_share_of_binders)
print("Discount change on high-ticket cluster: %.2fpp -> %.2fpp" % (
    binders_cluster.loc[0,'AvgDiscount_2016%'], binders_cluster.loc[0,'AvgDiscount_2017%']))
readme_check_6 = high_ticket_share_of_binders > 90
print("README claim reproduced:", "YES" if readme_check_6 else "NO")


High-ticket cluster share of total Binders decline: 96.6%
Discount change on high-ticket cluster: 12.50pp -> 38.00pp
README claim reproduced: YES


## Section 7 — Discount Analysis

**Business question:** Is discount depth associated with lower margin at the transaction level, and did average discount increase between the two periods?

**Method:** Compare average discount Q4-2016 vs Q4-2017; compare margin between high- and low-discount transactions.


In [20]:
avg_discount_16 = q4_16['Discount'].mean()*100
avg_discount_17 = q4_17['Discount'].mean()*100
print("Average Discount Q4-2016: %.2f%%" % avg_discount_16)
print("Average Discount Q4-2017: %.2f%%" % avg_discount_17)
print("Change: %.2f pp" % (avg_discount_17-avg_discount_16))


Average Discount Q4-2016: 14.69%
Average Discount Q4-2017: 15.88%
Change: 1.19 pp


In [21]:
q4_combined = pd.concat([q4_16, q4_17])
q4_combined['MarginRow'] = q4_combined['Profit']/q4_combined['Sales']
high_disc = q4_combined[q4_combined['Discount']>=0.3]['MarginRow']
low_disc = q4_combined[q4_combined['Discount']<0.3]['MarginRow']

discount_margin_table = pd.DataFrame({
    'Group': ['Discount >= 30%','Discount < 30%'],
    'N': [len(high_disc), len(low_disc)],
    'Median Margin': [high_disc.median(), low_disc.median()]
})
discount_margin_table


,Group,N,Median Margin
0,Discount >= 30%,292,-0.666667
1,Discount < 30%,1845,0.290000


In [22]:
readme_check_7 = (avg_discount_17 > avg_discount_16) and (high_disc.median() < low_disc.median())
print("README claim reproduced:", "YES" if readme_check_7 else "NO")


README claim reproduced: YES


## Section 8 — Central Region Analysis

**Business question:** Does the Central region show a stronger relationship between discount and margin than other regions?

**Method:** Compare margin change and revenue-share change for Central against the other three regions (already computed in Section 4); confirm Central shows the largest deterioration.


In [23]:
central_summary = region_mix_table.loc[['Central','East','South','West']].copy()
central_summary['Margin_change_pp'] = central_summary['Margin2017%'] - central_summary['Margin2016%']
central_summary['Share_change_pp'] = central_summary['Share2017%'] - central_summary['Share2016%']
central_summary.sort_values('Margin_change_pp')


,Share2016%,Share2017%,Margin2016%,Margin2017%,Margin_change_pp,Share_change_pp
Region,,,,,,
Central,28.834540,16.482669,28.662472,-4.678146,-33.340617,-12.351871
South,12.562036,20.019030,17.102137,-2.748183,-19.850321,7.456994
West,30.779154,28.496754,12.642837,13.652657,1.009820,-2.282400
East,27.824270,35.001547,6.648048,20.661666,14.013618,7.177277


In [24]:
most_negative_region = central_summary['Margin_change_pp'].idxmin()
print("Region with the largest margin deterioration:", most_negative_region)
readme_check_8 = most_negative_region == 'Central'
print("README claim reproduced:", "YES" if readme_check_8 else "NO")


Region with the largest margin deterioration: Central
README claim reproduced: YES


## Section 9 — Outlier Contribution

**Business question:** How much of the total decline is explained by a small number of large, deeply discounted transactions?

**Method:** Flag transactions with Profit below (mean - 3 standard deviations), computed on the full dataset; compare Q4-2016 vs Q4-2017 profit contribution from these flagged transactions.


In [25]:
mean_profit, std_profit = df['Profit'].mean(), df['Profit'].std()
outlier_threshold = mean_profit - 3*std_profit
outlier_ids = set(df[df['Profit'] < outlier_threshold]['Row ID'])

q4_16_out = q4_16[q4_16['Row ID'].isin(outlier_ids)]
q4_17_out = q4_17[q4_17['Row ID'].isin(outlier_ids)]

outlier_table = pd.DataFrame({
    'Period': ['Q4-2016','Q4-2017'],
    'N_outlier_rows': [len(q4_16_out), len(q4_17_out)],
    'Outlier_Profit': [q4_16_out['Profit'].sum(), q4_17_out['Profit'].sum()]
})
outlier_table


,Period,N_outlier_rows,Outlier_Profit
0,Q4-2016,2,-7538.2580
1,Q4-2017,8,-14037.4457


In [26]:
outlier_delta = q4_17_out['Profit'].sum() - q4_16_out['Profit'].sum()
outlier_share_of_decline = outlier_delta/total_delta*100
print("Outlier-attributable profit delta: $%.2f (%.1f%% of total decline)" % (outlier_delta, outlier_share_of_decline))

R16_x = q4_16[~q4_16['Row ID'].isin(outlier_ids)]['Sales'].sum()
P16_x = q4_16[~q4_16['Row ID'].isin(outlier_ids)]['Profit'].sum()
R17_x = q4_17[~q4_17['Row ID'].isin(outlier_ids)]['Sales'].sum()
P17_x = q4_17[~q4_17['Row ID'].isin(outlier_ids)]['Profit'].sum()
print("Margin excluding outliers -- 2016: %.2f%%  2017: %.2f%%" % (P16_x/R16_x*100, P17_x/R17_x*100))


Outlier-attributable profit delta: $-6499.19 (60.8% of total decline)
Margin excluding outliers -- 2016: 19.78%  2017: 15.84%


In [27]:
readme_check_9 = outlier_share_of_decline > 50
print("README claim reproduced:", "YES" if readme_check_9 else "NO")


README claim reproduced: YES


## Section 10 — Customer Segment Investigation

**Business question:** Is the apparent margin decline in the Corporate segment a systematic pattern, or driven by a small number of outlier transactions?

**Method:** Compute Corporate segment profit delta with and without the outlier transactions flagged in Section 9.


In [28]:
seg16 = q4_16.groupby('Segment').agg(Prof16=('Profit','sum'))
seg17 = q4_17.groupby('Segment').agg(Prof17=('Profit','sum'))
segment_delta = seg16.join(seg17)
segment_delta['ProfitDelta_gross'] = segment_delta['Prof17'] - segment_delta['Prof16']

seg16_x = q4_16[~q4_16['Row ID'].isin(outlier_ids)].groupby('Segment').agg(Prof16=('Profit','sum'))
seg17_x = q4_17[~q4_17['Row ID'].isin(outlier_ids)].groupby('Segment').agg(Prof17=('Profit','sum'))
segment_delta_ex = seg16_x.join(seg17_x)
segment_delta_ex['ProfitDelta_ex_outlier'] = segment_delta_ex['Prof17'] - segment_delta_ex['Prof16']

segment_table = segment_delta[['ProfitDelta_gross']].join(segment_delta_ex[['ProfitDelta_ex_outlier']])
segment_table


,ProfitDelta_gross,ProfitDelta_ex_outlier
Segment,,
Consumer,2656.6048,-889.5792
Corporate,-16045.7989,-7371.4812
Home Office,2698.0608,4069.1148


In [29]:
corp_gross = segment_table.loc['Corporate','ProfitDelta_gross']
corp_ex = segment_table.loc['Corporate','ProfitDelta_ex_outlier']
pct_explained_by_outliers = (1 - corp_ex/corp_gross) * 100
print("Corporate segment ProfitDelta (gross): $%.2f" % corp_gross)
print("Corporate segment ProfitDelta (excl. outliers): $%.2f" % corp_ex)
print("Share of Corporate decline explained by outlier transactions: %.1f%%" % pct_explained_by_outliers)

readme_check_10 = pct_explained_by_outliers > 50
print("README claim reproduced:", "YES" if readme_check_10 else "NO")


Corporate segment ProfitDelta (gross): $-16045.80
Corporate segment ProfitDelta (excl. outliers): $-7371.48
Share of Corporate decline explained by outlier transactions: 54.1%
README claim reproduced: YES


## Summary of Reproduction Checks

All ten sections above were checked against the corresponding README claim. See the Traceability Matrix in the project root for the full mapping between README statements and notebook outputs.

Validation of these findings against alternative aggregation levels, subgroups, and confounding variables is performed separately in `03_Validation.ipynb`.
